# Author a Classification Task

Notebook 01 gives the guided corpus workflow. This notebook focuses on the task definition: first use a built-in classifier, then define a new label vocabulary as data. Classification yields comparable per-paper labels; notebook 06's miner yields concrete items to verify.


In [ ]:
import sys
from pathlib import Path

if (Path.cwd() / "notebooks" / "epilens_nb.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "notebooks"))
import epilens_nb as nb
WORK_DIR = nb.bootstrap()


## Prepare a small offline index

The helper provides two short sample papers and a deterministic keyword embedder. The index is scratch data for this notebook, so no model download or API key is needed.


In [ ]:
from epilens.workflows import PaperClassifier
from epilens.workflows.classification import DataAccessibilityClassifierConfig

papers = nb.classification_papers()
embedder = nb.TinyKeywordEmbedder(nb.CLASSIFICATION_VOCABULARY)
strategy_name = "task-authoring-demo"
academic_db = nb.build_academic_db(papers, strategy_name)
_, retriever = nb.build_index(
    WORK_DIR / "index", papers, embedder, chunk_size=650, chunk_overlap=75
)


## Use a built-in scheme

`DataAccessibilityClassifierConfig` labels **what the paper reports** about access to data; it does not test whether a request or link works. The fixed JSON below illustrates the workflow and does not measure classification accuracy.


In [ ]:
access_classifier = PaperClassifier(
    retriever=retriever,
    generator=nb.FixedJSONGenerator({
        "classification": ["D"],
        "primary_label": "D",
        "reasoning": "The paper says annotations are available from the author on request.",
    }),
    strategy_name=strategy_name,
    config=DataAccessibilityClassifierConfig(top_k=4),
    academic_db=academic_db,
)
access = access_classifier.run_detailed("paper_methods")
print("Reported access:", [label.value for label in access.decision.result.classification])
for chunk in access.decision.top_evidence:
    if chunk.section_title == "Data availability":
        print("Supporting passage:", chunk.text)


## Define your own labels

A classifier `TaskSpec` holds the category codes, definitions, and retrieval examples. `multi_label=False` asks for one primary route. The fallback is a review queue, not a verified judgment.


In [ ]:
from epilens.workflows.registry import TaskSpec, build_classifier_config_from_spec

route_spec = TaskSpec.model_validate({
    "key": "review_route",
    "kind": "classifier",
    "label": "Review route",
    "multi_label": False,
    "default_label": "manual_review",
    "labels": [
        {
            "code": "action", "name": "Action or policy",
            "definition": "Main contribution directly informs an action or policy.",
            "examples": ["The findings support a hospital guideline."],
        },
        {
            "code": "method", "name": "Method or tool",
            "definition": "Main contribution develops or evaluates a reusable method.",
            "examples": ["We introduce a reusable screening tool."],
        },
        {
            "code": "manual_review", "name": "Manual review",
            "definition": "Evidence is insufficient or ambiguous.",
            "examples": ["The paper's main contribution is unclear."],
        },
    ],
})
route_config = build_classifier_config_from_spec(route_spec)
print(route_config.category_labels)


In [ ]:
route_classifier = PaperClassifier(
    retriever=retriever,
    generator=nb.FixedJSONGenerator({
        "classification": ["method"],
        "reasoning": "The main contribution is a reusable screening method.",
    }),
    strategy_name=strategy_name,
    config=route_config,
    academic_db=academic_db,
)
route = route_classifier.run_detailed("paper_methods")
print("Review route:", route.decision.result.classification)
print("Evidence chunks:", len(route.decision.top_evidence))


## Before corpus use

The fixed generator makes both labels illustrative. For real classification, use an `LLMGenerator`, inspect the retrieved passages, and check the scheme against manually labelled papers from the target corpus. Schema validation ensures a usable output shape; evidence links make decisions inspectable. Neither establishes that a label is correct. Save `route_spec.model_dump()` as JSON to use this scheme through `epilens classify --task-file` or a workspace task file.
